# ดูทุกแถวและทุก feature ของ dataset

เลือกไฟล์ใน cell แรกแล้วรันตามลำดับ. ใช้ชื่อใน `DATASETS` หรือ path ของ CSV/Parquet ภายใน repository เพื่อดู artifact ทดลองอื่น. `df` คือไฟล์เต็มทุกแถว; ใน VS Code เปิด Variables → `df` → **View as DataFrame** เพื่อเลื่อน/กรองข้อมูล. รายละเอียดที่มาของไฟล์หลักอยู่ใน [คู่มือ](07_ดูหน้าตา_dataset.md).


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATASETS = {
    "profiles_cleaned": "data/processed/all_profiles_cleaned.csv",
    "profiles_with_images": "data/final/profiles_with_images.csv",
    "image_features": "data/final/image_features_complete.csv",
    "labeled_pairs": "data/processed/labeled_pairs.csv",
    "feature_matrix": "data/processed/feature_matrix.csv",
    "train_dataset": "data/processed/train_dataset.csv",
    "test_dataset": "data/processed/test_dataset.csv",
    "candidate_pairs": "data/processed/candidate_pairs.csv",
    "predictions": "data/processed/predictions.csv",
}
DATASET = "feature_matrix"  # ชื่อใน DATASETS หรือ path เช่น experiments/.../features/gb17.parquet
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/processed/feature_matrix.csv").exists()), None)
if root is None:
    raise FileNotFoundError("เปิด notebook จากโฟลเดอร์ใน repository นี้")
path = (root / DATASETS.get(DATASET, DATASET)).resolve()
if not path.is_relative_to(root.resolve()) or path.suffix.lower() not in {".csv", ".parquet"} or not path.is_file():
    raise ValueError("เลือกไฟล์ CSV/Parquet ที่มีอยู่ภายใน repository")
df = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path, low_memory=False)
print(f"{path.relative_to(root)}: {len(df):,} rows × {len(df.columns)} columns")


data/processed/feature_matrix.csv: 204,701 rows × 21 columns


## รายชื่อไฟล์ข้อมูลทั้งหมดในโครงการ

รายการนี้แสดงชื่อและขนาดไฟล์ CSV/Parquet โดยไม่อ่านเนื้อหา เลือก path ไปใส่ใน `DATASET` แล้วรัน notebook ใหม่. ไฟล์ต่าง run/split ต้องอ่านแยกตาม protocol.


In [2]:
files = sorted(
    p for folder in ("data", "train_data", "experiments", "artifacts")
    for p in (root / folder).rglob("*") if p.is_file() and p.suffix.lower() in {".csv", ".parquet"}
)
file_catalog = pd.DataFrame({
    "path": [str(p.relative_to(root)) for p in files],
    "size_MB": [round(p.stat().st_size / 1024**2, 2) for p in files],
})
print(f"พบ {len(file_catalog):,} ไฟล์")
display(file_catalog)


พบ 512 ไฟล์


,path,size_MB
0,artifacts/audit/icsec_review_20261010/full2m_s...,0.00
1,artifacts/audit/icsec_review_20261010/paired_c...,0.00
2,artifacts/audit/icsec_review_20261010/role_pri...,0.00
3,artifacts/decisioning/analysis_decision_matrix...,1.76
4,artifacts/decisioning/analysis_decision_matrix...,1.22
...,...,...
507,train_data/comparison_location_full_comparison...,1.18
508,train_data/ground_truth_labels.csv,1.04
509,train_data/ground_truth_matrix_index.csv,1.02
510,train_data/ground_truth_pairs.csv,2.15


## ทุกคอลัมน์

ตารางนี้เรียงตามลำดับคอลัมน์ในไฟล์ แสดงชนิดข้อมูล จำนวนค่าว่าง และจำนวนค่าที่แตกต่างกัน. คอลัมน์แบบข้อความที่มีค่ามากอาจใช้เวลาคำนวณเล็กน้อย.


In [3]:
columns = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).to_numpy(),
    "missing": df.isna().sum().to_numpy(),
    "missing_%": (df.isna().mean() * 100).round(2).to_numpy(),
    "unique": df.nunique(dropna=True).to_numpy(),
})
display(columns)


,column,dtype,missing,missing_%,unique
0,profile_id_a,str,0,0.00,35926
1,profile_id_b,str,0,0.00,24903
2,entity_id_a,str,15,0.01,15176
3,label,int64,0,0.00,2
4,username_jaro,float64,0,0.00,356
5,username_token_sort,float64,0,0.00,349
6,username_levenshtein,float64,0,0.00,349
7,username_both_empty,float64,0,0.00,1
8,fullname_jaro,float64,0,0.00,591
9,fullname_token_sort,float64,0,0.00,525


## หน้าตาแต่ละแถว

แสดงต้นและท้ายไฟล์อย่างละ 5 แถว. ตัวแปร `df` ข้างบนยังเก็บทุกแถว และ `df.iloc[100:120]` ใช้ดูช่วงแถวที่ต้องการได้.


In [4]:
with pd.option_context("display.max_columns", None, "display.max_colwidth", 70, "display.width", 180):
    display(df.head(5))
    display(df.tail(5))


,profile_id_a,profile_id_b,entity_id_a,label,username_jaro,username_token_sort,username_levenshtein,username_both_empty,fullname_jaro,fullname_token_sort,fullname_levenshtein,fullname_both_empty,bio_jaro,bio_token_sort,bio_levenshtein,bio_both_empty,bio_tfidf_cosine,url_exact_match,url_domain_match,same_platform,location_sim
0,googleplus_fefejohnson,instagram_victorrubencp,fefemakeupartistry,0,0.166667,0.166667,0.166667,0.0,0.342857,0.250000,0.250000,0.0,0.400000,0.157895,0.157895,0.0,0.0,0.0,0.0,0.0,1.00
1,googleplus_joshfinnie,twitter_francovanvuuren,joshfinnie,0,0.320000,0.320000,0.320000,0.0,0.327273,0.285714,0.285714,0.0,0.407368,0.331683,0.331683,0.0,0.0,0.0,0.0,0.0,0.45
2,googleplus_ginaschreck,twitter_ginaschreck,ginaschreck,1,1.000000,1.000000,1.000000,0.0,0.900000,0.600000,0.600000,0.0,0.469565,0.397661,0.397661,0.0,0.0,0.0,0.0,0.0,0.30
3,googleplus_danielapoetyca,instagram_deepeshmd,poetyca,0,0.347826,0.347826,0.347826,0.0,0.294118,0.294118,0.294118,0.0,0.400000,0.105263,0.105263,0.0,0.0,0.0,0.0,0.0,1.00
4,googleplus_tonychau,instagram_tonydiehl,tonychau,0,0.588235,0.588235,0.588235,0.0,0.631579,0.631579,0.631579,0.0,0.400000,0.071429,0.071429,0.0,0.0,0.0,0.0,0.0,1.00


,profile_id_a,profile_id_b,entity_id_a,label,username_jaro,username_token_sort,username_levenshtein,username_both_empty,fullname_jaro,fullname_token_sort,fullname_levenshtein,fullname_both_empty,bio_jaro,bio_token_sort,bio_levenshtein,bio_both_empty,bio_tfidf_cosine,url_exact_match,url_domain_match,same_platform,location_sim
204696,googleplus_chelwolverton,twitter_courtneywalsh,chelw,0,0.384615,0.384615,0.384615,0.0,0.357143,0.357143,0.357143,0.0,0.383824,0.291429,0.291429,0.0,0.0,0.0,0.0,0.0,0.30
204697,twitter_dparker007,instagram_ramisoj555,DParker007,0,0.100000,0.100000,0.100000,0.0,0.450000,0.333333,0.333333,0.0,0.360656,0.360656,0.360656,0.0,0.0,0.0,0.0,0.0,0.60
204698,twitter_jisforjoe,instagram_isaiassharon,jisforJoe,0,0.380952,0.380952,0.380952,0.0,0.315789,0.315789,0.315789,0.0,0.385321,0.385321,0.385321,0.0,0.0,0.0,0.0,0.0,0.36
204699,twitter_digitalsista,instagram_julienhorn,digitalsista,0,0.181818,0.181818,0.181818,0.0,0.317647,0.307692,0.307692,0.0,0.406250,0.406250,0.406250,0.0,0.0,0.0,0.0,0.0,1.00
204700,googleplus_jamesrobinson,instagram_jamesbergin,jamesbrobinson,0,0.666667,0.666667,0.666667,0.0,0.692308,0.692308,0.692308,0.0,0.399057,0.385057,0.385057,0.0,0.0,0.0,0.0,0.0,1.00


## การกระจายของ label / platform / สถานะภาพ

แสดงเฉพาะคอลัมน์ที่มีอยู่ในไฟล์ที่เลือก.


In [5]:
for column in ("label", "platform", "platform_a", "platform_b", "has_image", "decision"):
    if column in df.columns:
        print(f"\n{column}")
        display(df[column].value_counts(dropna=False).rename("rows").to_frame())



label


,rows
label,
0,175458
1,29243
